# Application Programming Interface

In [ ]:
'''
- Backend Application
- API 
    - Status Code
        - 404, 500
    - JSON Data
- User Request for Data
    - Get -> Read Data
    - Post -> Create / Insert
    - Put -> Update
    - Delete -> Remove
'''

In [2]:
import requests
import os
import time

## API Authentication

In [17]:
# Token -> Bearer <Token>
url = "https://httpbin.org/bearer"

login_token = os.getenv('API_TOKEN', 'demo123')

authentication_parms = {
    'Authorization': f'Bearer {login_token}',
    'content_type': 'application/json'
}

try:
    response = requests.get(url, headers=authentication_parms, timeout=140)
    print(f'Login Status Code: {response.status_code}\nToken: {response.json()}')
    print('Login Succesfull')
except requests.exceptions.RequestException as e:
    print(f'{response.status_code}')

Login Status Code: 200
Token: {'authenticated': True, 'token': 'demo123'}
Login Succesfull


## Pagination and Limit

In [43]:
API_URL = 'https://jsonplaceholder.typicode.com/posts'

page = 1
max_page_to_fetch = 5
all_paginated_data = []

In [44]:
while page <= max_page_to_fetch:
    page_params = {
        '_page': page,
        '_limit': max_page_to_fetch
    }

    try:
        response = requests.get(API_URL, params=page_params)

        if response.status_code != 200:
            print(f'Error: {response.status_code}')
            break

        page_data = response.json()
        if not page_data:
            break

        all_paginated_data.extend(page_data)
        print(f'Page {page} fetched. Stored {len(page_data)} records.')
        page += 1
        time.sleep(1)
        
    except requests.exceptions.RequestException:
        print(f'Error: {response.status_code}')

Page 1 fetched. Stored 5 records.
Page 2 fetched. Stored 5 records.
Page 3 fetched. Stored 5 records.
Page 4 fetched. Stored 5 records.
Page 5 fetched. Stored 5 records.


## Fetch API Data and Merge with Local Data

In [61]:
import pandas as pd

local_users = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'department': ['Sales', 'Marketing', 'HR', 'IT', 'Support', 'Sales', 'Marketing', 'HR', 'IT', 'Support']
}

local_df = pd.DataFrame(local_users)
local_df.to_csv('users.csv', index=False)

user_df = pd.read_csv('users.csv')
user_df.head(2)

,id,department
0,1,Sales
1,2,Marketing


In [66]:
USER_API = 'https://jsonplaceholder.typicode.com/users'

try:
    response = requests.get(USER_API)

    if response.status_code == 200:
        user_data = response.json()

        user_json = pd.json_normalize(user_data)
        user_columns = user_json[['id', 'name', 'username', 'phone', 'address.street', 'company.name']]

        combined_df = pd.merge(user_columns, user_df, on='id', how='inner')
        combined_df = combined_df.rename(columns = {
            'address.street': 'street', 'company.name': 'company name'
        })
    else:
        print(f'Error: {response.status_code}')
except requests.exceptions.RequestException:
    print(f'Error: {response.status_code}')

In [69]:
combined_df.to_csv('users.csv', index=False)